# 03 · NeuMF-pre on MovieLens-1M (stand-alone run)

Thesis §5.7.1 · Table 16 · Figures 26–27

Stand-alone version of the NeuMF-pre experiment: GMF is pretrained for 3 epochs and MLP for 12 epochs,
their best validation epochs in notebook 02. Both are loaded into NeuMF (α = 0.5), fine-tuned for
30 epochs, and the best validation checkpoint is evaluated on the test split.

In [ ]:
# Folder containing ml-1m.ratings.dat and yelp.rating (see data/README.md).
# On Kaggle, point this at the attached dataset, e.g. "/kaggle/input/<dataset-name>".
DATA_DIR = "../data"

In [ ]:
#Plotting training and Validation accuracy
def plot_accuracy(history):
    plt.figure()
    acc_key = 'accuracy' if 'accuracy' in history.history else 'binary_accuracy'
    val_acc_key = 'val_' + acc_key

    plt.plot(history.history[acc_key], label='Train Accuracy')
    if val_acc_key in history.history:
        plt.plot(history.history[val_acc_key], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()



#HR (hit ratio)
def HR(ranked_predicted_list, true_test_item, number_of_rec_items = 10):
    top_rec_items = ranked_predicted_list[:number_of_rec_items]
    item_ids = [item_id for item_id, _ in top_rec_items]
    if true_test_item in item_ids:
        return 1.0
    else:
        return 0.0
#Normalized Discounted Cumulative Gain (NDCG) 


def NDCG(ranked_predicted_list, true_test_item, number_of_rec_items):
    top_rec_items = ranked_predicted_list[:number_of_rec_items]
    item_ids = [item_id for item_id, _ in top_rec_items]
    for rank, item_id in enumerate(item_ids):
        if item_id == true_test_item:
            return 1.0/math.log2((rank+1) +1)
    return 0



#Get the predictions


#Evaluation using HR
def eval_hr_ndcg(
    interaction_matrix,        # kept only for API compatibility, not used
    interaction_array,         # shape: [num_users, num_items]
    test_df,                   # label-encoded user_id, item_id
    model,
    number_of_rec_items,
    max_test_size,
    neg_sample_size=100,
    eval_batch_size=4096,      # new: batch size for prediction
):
    """
    Faster evaluation of HR@K and NDCG@K:

    - Precomputes user -> positive item set from interaction_array.
    - For each test (u, i), samples 'neg_sample_size' negatives by random sampling
      (similar to training), avoiding all positives and the test item.
    - Builds one big batch of (user, item) pairs and calls model.predict once.
    - For each user, sorts their candidates, looks at top-K, and computes HR/NDCG.
    """
    import math
    import numpy as np
    import random

    num_users, num_items = interaction_array.shape

    # 1) Precompute user -> set(positive items) once
    user_pos_items = [
        set(np.where(interaction_array[u] == 1)[0])
        for u in range(num_users)
    ]

    users_test = test_df['user_id'].values
    items_test = test_df['item_id'].values

    # Containers for batched prediction
    batch_user_ids = []
    batch_item_ids = []
    sample_slices = []  # list of (start_idx, end_idx, true_item_id)

    num_eval_users = 0

    # 2) Build candidate lists (negatives + 1 positive) for each (u, i)
    for u, i in zip(users_test, items_test):
        # safety: skip if out of bounds
        if u < 0 or u >= num_users or i < 0 or i >= num_items:
            continue

        user_idx = int(u)
        item_idx = int(i)

        pos_items = user_pos_items[user_idx]

        # maximum possible negatives for this user (all items minus positives minus this test item)
        max_possible_neg = num_items - len(pos_items) - (0 if item_idx in pos_items else 1)
        if max_possible_neg <= 0:
            # nothing to sample from
            continue

        n_neg = min(neg_sample_size, max_possible_neg)

        # Sample negatives by random integer, rejecting positives / test item / duplicates
        neg_items = []
        neg_set = set()
        while len(neg_items) < n_neg:
            j = random.randint(0, num_items - 1)
            if (j not in pos_items) and (j != item_idx) and (j not in neg_set):
                neg_items.append(j)
                neg_set.add(j)

        candidate_list = np.array(neg_items + [item_idx], dtype=np.int32)

        start = len(batch_item_ids)
        end = start + candidate_list.size

        batch_user_ids.extend([user_idx] * candidate_list.size)
        batch_item_ids.extend(candidate_list.tolist())
        sample_slices.append((start, end, item_idx))

        num_eval_users += 1
        if num_eval_users >= max_test_size:
            break

    # No valid users
    if num_eval_users == 0:
        print(f"HR@{number_of_rec_items} = 0.0000")
        print(f"NDCG@{number_of_rec_items} = 0.0000")
        return 0.0, 0.0

    batch_user_ids = np.asarray(batch_user_ids, dtype=np.int32)
    batch_item_ids = np.asarray(batch_item_ids, dtype=np.int32)

    # 3) Single batched prediction
    preds = model.predict(
        [batch_user_ids, batch_item_ids],
        batch_size=eval_batch_size,
        verbose=0
    ).reshape(-1)

    # 4) Compute HR@K and NDCG@K
    num_hits = 0.0
    sum_ndcg = 0.0

    for start, end, true_item in sample_slices:
        scores = preds[start:end]
        items_slice = batch_item_ids[start:end]

        # sort candidate scores in descending order
        order = np.argsort(-scores)
        topk_order = order[:number_of_rec_items]
        topk_items = items_slice[topk_order]

        # if true_item is in top-K, compute rank and NDCG
        match_positions = np.where(topk_items == true_item)[0]
        if match_positions.size > 0:
            rank = int(match_positions[0])  # 0-based within top-K
            num_hits += 1.0
            # NDCG for single relevant item at position 'rank'
            sum_ndcg += 1.0 / math.log2(rank + 2)

    hr = num_hits / num_eval_users
    ndcg = sum_ndcg / num_eval_users

    print(f"HR@{number_of_rec_items} = {hr:.4f}")
    print(f"NDCG@{number_of_rec_items} = {ndcg:.4f}")

    return hr, ndcg

    
   
    
#Get weights from pretrained models
def get_weights(gmf_model, mlp_model,neumf_model,hidden_units,alpha):
    
    #From GMF (emb layer):
    gmf_user_emb = gmf_model.get_layer('gmf_user_embedding').get_weights()
    gmf_item_emb = gmf_model.get_layer('gmf_item_embedding').get_weights()

    #From GMF (output layer)
    gmf_output_weights = gmf_model.get_layer('outputs').get_weights()  
        
    #To NeuMF from GMF
    #(Emb)
    neumf_model.get_layer('gmf_user_embedding').set_weights(gmf_user_emb)
    neumf_model.get_layer('gmf_item_embedding').set_weights(gmf_item_emb)

    #(Output)

    #neumf_model.get_layer('gmf_outputs').set_weights(gmf_output_weights)

    #From MLP (emb layer)
    mlp_user_emb = mlp_model.get_layer('user_embedding').get_weights()
    mlp_item_emb = mlp_model.get_layer('item_embedding').get_weights()
    #Fron MLP (layers)
    
    #From MLP (output layer)
    mlp_output_weights = mlp_model.get_layer('outputs').get_weights()  

    #To NeuMF from GMF
    #(Emb)
    neumf_model.get_layer('mlp_user_embedding').set_weights(mlp_user_emb)
    neumf_model.get_layer('mlp_item_embedding').set_weights(mlp_item_emb)
    #(Layers)
    for idx, _ in enumerate(hidden_units):
        mlp_layer_weights = mlp_model.get_layer(f"hidden_{idx}").get_weights()
        neumf_model.get_layer(f"mlp_hidden_{idx}").set_weights(mlp_layer_weights)
    #(Output)
    #neumf_model.get_layer('mlp_outputs').set_weights(mlp_output_weights)

    #Concatenate weights:
    Wg, bg = gmf_output_weights
    Wm, bm = mlp_output_weights
    W_final = np.concatenate([alpha * Wg, (1 - alpha) * Wm], axis=0)
    b_final = alpha * bg + (1 - alpha) * bm
    #Add concatenated weights and biases to the NEUMF
    neumf_model.get_layer('neumf_outputs').set_weights([W_final, b_final])
    
    print("Weights added")
    

    
    





#Used libraries

import numpy as np
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Multiply, 
    Dense, Concatenate, Activation
)
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from collections import defaultdict
import random
import math
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os
import sys

df = pd.read_csv(f"{DATA_DIR}/ml-1m.ratings.dat", sep="::", engine="python",
                 names=["user_id", "item_id", "rating", "timestamp"])
print("Original data:")
print(df.head())
# -----------------------------
# 1. Basic sanity check
# -----------------------------
user_counts = df.groupby("user_id").size()
print("Minimum number of ratings per user:", user_counts.min())

# -----------------------------
# 2. Leave-one-out split: train / test
# -----------------------------
# Sort by user, then timestamp
df = df.sort_values(['user_id', 'timestamp'])

train_list = []
test_list  = []

for user_id, group in df.groupby('user_id'):
    user_history = group.values  # or group.to_numpy()

    # Last interaction = test
    test_interaction = user_history[-1]
    # All previous = train
    train_interactions = user_history[:-1]

    for row in train_interactions:
        train_list.append(row)
    test_list.append(test_interaction)

train_df = pd.DataFrame(train_list, columns=df.columns).reset_index(drop=True)
test_df  = pd.DataFrame(test_list,  columns=df.columns).reset_index(drop=True)

# -----------------------------
# 3. Validation split from train (second last per user)
# -----------------------------
train_df = train_df.sort_values(['user_id', 'timestamp'])
val_df = train_df.groupby('user_id').tail(1).copy()      # 1 row per user
train_df_cv = train_df.drop(val_df.index).copy()         # remaining as train

train_df = train_df_cv.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Training size:", len(train_df))
print("Validation size:", len(val_df))
print("Testing size: ", len(test_df))

# -----------------------------
# 4. Label-encode user/item on TRAIN + VAL + TEST
# -----------------------------
ue, ie = LabelEncoder(), LabelEncoder()

all_ids = pd.concat([
    train_df[['user_id', 'item_id']],
    val_df[['user_id', 'item_id']],
    test_df[['user_id', 'item_id']]
], axis=0)

ue.fit(all_ids['user_id'])
ie.fit(all_ids['item_id'])

train_df['user_id'] = ue.transform(train_df['user_id'])
train_df['item_id'] = ie.transform(train_df['item_id'])
val_df['user_id']   = ue.transform(val_df['user_id'])
val_df['item_id']   = ie.transform(val_df['item_id'])
test_df['user_id']  = ue.transform(test_df['user_id'])
test_df['item_id']  = ie.transform(test_df['item_id'])

num_users = len(ue.classes_)
num_items = len(ie.classes_)

# -----------------------------
# 5. Build per-user positive sets from TRAIN ONLY
#    (to avoid leakage when sampling negatives)
# -----------------------------
user_pos = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
all_items = np.arange(num_items, dtype=np.int32)

# -----------------------------
# 6. Negative sampling for NeuMF training
# -----------------------------
num_neg = 4  # how many negatives per positive

user_input = []
item_input = []
labels     = []

# Go through each positive in the TRAIN set
for row in train_df.itertuples(index=False):
    u = row.user_id
    i = row.item_id

    # Positive instance
    user_input.append(u)
    item_input.append(i)
    labels.append(1)

    # Negative instances
    pos_items = user_pos[u]
    for _ in range(num_neg):
        j = np.random.randint(num_items)
        # resample until we get an item not interacted with by this user
        while j in pos_items:
            j = np.random.randint(num_items)
        user_input.append(u)
        item_input.append(j)
        labels.append(0)

user_input_train = np.array(user_input, dtype=np.int32)
item_input_train = np.array(item_input, dtype=np.int32)
labels_train     = np.array(labels,     dtype=np.float32)

print("Train triplets (pos+neg):", len(labels_train))
print("Num users:", num_users, "Num items:", num_items)


In [ ]:
#GMF model

def build_GMF(num_users, num_items, learning_rate,emb_size):
    user_input = Input(shape=(1,), name='user_input')
    item_input = Input(shape=(1,), name='item_input')
    
    user_embedding = Embedding(
        input_dim=num_users, 
        output_dim=emb_size, 
        name='gmf_user_embedding'
    )
    item_embedding = Embedding(
        input_dim=num_items,
        output_dim=emb_size,
        name='gmf_item_embedding'
    )
    
    user_latent = Flatten()(user_embedding(user_input))
    item_latent = Flatten()(item_embedding(item_input))
    gmf_vector = Multiply()([user_latent, item_latent])
    
    outputs = Dense(1, activation='sigmoid', name='outputs')(gmf_vector)
    model = Model(
        inputs=[user_input,item_input],
        outputs=outputs
    )
    
    return model

#MPL model
def build_MLP(num_users, num_items, emb_size, hidden_units, learning_rate):
    user_input = Input(shape=(1,), name='user_input')   
    item_input = Input(shape=(1,), name='item_input')   

    user_embedding = Embedding(
        input_dim=num_users,
        output_dim=emb_size,
        name='user_embedding'
    )(user_input)
    user_embedding = Flatten()(user_embedding)

    item_embedding = Embedding(
        input_dim=num_items,
        output_dim=emb_size,
        name='item_embedding'
    )(item_input)
    item_embedding = Flatten()(item_embedding)

    concat_vec = Concatenate(axis=1)([user_embedding, item_embedding])
    x = concat_vec
    
    for idx, units in enumerate(hidden_units):
        x = Dense(units, activation='relu', name=f"hidden_{idx}")(x)
        
    outputs = Dense(1, activation='sigmoid', name='outputs')(x)
    
    model = Model(
        inputs=[user_input, item_input],
        outputs=outputs
    )
    return model


#NeuMF model

def build_NEUMF(
    num_users,
    num_items,
    latent_dim,      # currently unused, kept for API compatibility
    hidden_units,
    emb_size,
    learning_rate,
    alpha=0.5        # currently unused here; used in get_weights
):
    user_input = Input(shape=(1,), name='user_input')   
    item_input = Input(shape=(1,), name='item_input') 

    # ----- MLP branch -----
    mlp_user_embedding = Embedding(
        input_dim=num_users,
        output_dim=emb_size,
        name='mlp_user_embedding'
    )(user_input)
    mlp_user_embedding = Flatten()(mlp_user_embedding)

    mlp_item_embedding = Embedding(
        input_dim=num_items,
        output_dim=emb_size,
        name='mlp_item_embedding'
    )(item_input)
    mlp_item_embedding = Flatten()(mlp_item_embedding)

    mlp_concat_vec = Concatenate(axis=1)([mlp_user_embedding, mlp_item_embedding])
    x = mlp_concat_vec
    for idx, units in enumerate(hidden_units):
        x = Dense(units, activation='relu', name=f"mlp_hidden_{idx}")(x)
    mlp_vector = x

    # ----- GMF branch -----
    gmf_user_embedding = Embedding(
        input_dim=num_users, 
        output_dim=emb_size, 
        name='gmf_user_embedding'
    )
    gmf_item_embedding = Embedding(
        input_dim=num_items,
        output_dim=emb_size,
        name='gmf_item_embedding'
    )
    gmf_user_latent = Flatten()(gmf_user_embedding(user_input))
    gmf_item_latent = Flatten()(gmf_item_embedding(item_input))
    gmf_vector = Multiply()([gmf_user_latent, gmf_item_latent])

    # ----- NeuMF fusion -----
    neumf_concat = Concatenate(axis=1)([mlp_vector, gmf_vector])
    neumf_outputs = Dense(1, activation='sigmoid', name='neumf_outputs')(neumf_concat)

    model = Model(
        inputs=[user_input, item_input],
        outputs=neumf_outputs
    )
    # DO NOT compile here; you compile outside
    return model


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ---- Build interaction matrix from TRAIN_CV and align to full id space ----
val_inter_mat = (
    train_df.assign(interaction=1)
            .pivot_table(index='user_id', columns='item_id',
                         values='interaction', fill_value=0)
)

val_inter_mat = val_inter_mat.reindex(index=np.arange(num_users),
                                      columns=np.arange(num_items),
                                      fill_value=0)
val_inter_arr = val_inter_mat.values

# -----------------------------
# Hyperparameters
# -----------------------------
GMF_PRE_EPOCHS = 3      # GMF pretraining epochs
MLP_PRE_EPOCHS = 12      # MLP pretraining epochs
NEUMF_EPOCHS   = 30     # NeuMF fine-tuning epochs

BATCH_SIZE     = 256
K_val          = 10     # K for validation HR/NDCG
emb_size       = 64
hidden_units   = [32, 16, 8]
latent_dim     = 32     # for build_NEUMF API compatibility
learning_rate  = 0.001
alpha          = 0.5    # blend factor in get_weights for final layer

# ============================================================
# 1) PRETRAIN GMF FOR 3 EPOCHS
# ============================================================
gmf_model = build_GMF(
    num_users=num_users,
    num_items=num_items,
    learning_rate=learning_rate,
    emb_size=emb_size
)

gmf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['binary_accuracy']
)

print("\n=== Pretraining GMF for 3 epochs ===")
gmf_model.fit(
    x=[user_input_train, item_input_train],
    y=labels_train,
    batch_size=BATCH_SIZE,
    epochs=GMF_PRE_EPOCHS,
    shuffle=True,
    validation_split=0.0,  # no internal Keras val; real val via HR/NDCG later
    verbose=1
)

# ============================================================
# 2) PRETRAIN MLP FOR 6 EPOCHS
# ============================================================
mlp_model = build_MLP(
    num_users=num_users,
    num_items=num_items,
    emb_size=emb_size,
    hidden_units=hidden_units,
    learning_rate=learning_rate
)

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['binary_accuracy']
)

print("\n=== Pretraining MLP for 6 epochs ===")
mlp_model.fit(
    x=[user_input_train, item_input_train],
    y=labels_train,
    batch_size=BATCH_SIZE,
    epochs=MLP_PRE_EPOCHS,
    shuffle=True,
    validation_split=0.0,  # same idea: we evaluate with HR/NDCG separately
    verbose=1
)

# ============================================================
# 3) BUILD NEUMF & LOAD PRETRAINED GMF + MLP WEIGHTS
# ============================================================
neumf_model = build_NEUMF(
    num_users=num_users,
    num_items=num_items,
    latent_dim=latent_dim,
    hidden_units=hidden_units,
    emb_size=emb_size,
    learning_rate=learning_rate,
    alpha=alpha
)

# transfer GMF + MLP weights into NeuMF
get_weights(gmf_model, mlp_model, neumf_model, hidden_units, alpha)

# compile NeuMF for fine-tuning
neumf_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['binary_accuracy']
)

print("\nSanity check (NeuMF with GMF 3ep + MLP 6ep pretrained):")
print("train_df shape:", train_df.shape)
print("num_users, num_items:", num_users, num_items)
print("train triples:", len(labels_train))

# ============================================================
# 4) TRAIN NEUMF WITH PERIODIC HR/NDCG VALIDATION
# ============================================================
neumf_val_hr = []
neumf_val_ndcg = []
neumf_train_loss_hist = []
neumf_train_acc_hist  = []
neumf_best = {'ndcg': -1.0, 'hr': 0.0, 'epoch': -1, 'weights': None}

for ep in range(1, NEUMF_EPOCHS + 1):
    print(f"\n=== NeuMF (pretrained) Epoch {ep}/{NEUMF_EPOCHS} ===")
    
    hist = neumf_model.fit(
        x=[user_input_train, item_input_train],
        y=labels_train,
        batch_size=BATCH_SIZE,
        epochs=1,
        shuffle=True,
        verbose=1
    )
    
    neumf_train_loss_hist.append(float(hist.history['loss'][0]))
    if 'binary_accuracy' in hist.history:
        neumf_train_acc_hist.append(float(hist.history['binary_accuracy'][0]))
    elif 'accuracy' in hist.history:
        neumf_train_acc_hist.append(float(hist.history['accuracy'][0]))
    else:
        neumf_train_acc_hist.append(np.nan)
    
    # every 3 epochs: full validation HR/NDCG at K_val
    if ep % 3 == 0:
        hr, ndcg = eval_hr_ndcg(
            val_inter_mat,
            val_inter_arr,
            val_df,
            neumf_model,
            number_of_rec_items=K_val,
            max_test_size=10_000,
            neg_sample_size=100
        )
        neumf_val_hr.append(hr)
        neumf_val_ndcg.append(ndcg)
        
        print(f"[VAL] NeuMF-pre Epoch {ep}: HR@{K_val}={hr:.4f}, NDCG@{K_val}={ndcg:.4f}")
        
        if ndcg > neumf_best['ndcg']:
            neumf_best['ndcg'] = ndcg
            neumf_best['hr'] = hr
            neumf_best['epoch'] = ep
            neumf_best['weights'] = neumf_model.get_weights()
            print(f"New best NeuMF-pre epoch: {ep} (NDCG={ndcg:.4f}, HR={hr:.4f})")

print("NeuMF-pre VAL HR history:", neumf_val_hr)
print("NeuMF-pre VAL NDCG history:", neumf_val_ndcg)

# ============================================================
# 5) FINAL TEST @ K = 5, 10, 20
# ============================================================
if neumf_best['weights'] is not None:
    neumf_model.set_weights(neumf_best['weights'])
    print(f"\nLoaded best NeuMF-pre weights from epoch {neumf_best['epoch']}")
else:
    print("\nWarning: no best NeuMF-pre checkpoint stored, using last-epoch weights")

for K_test in [5, 10, 20]:
    hr_t, ndcg_t = eval_hr_ndcg(
        val_inter_mat,
        val_inter_arr,
        test_df,
        neumf_model,
        number_of_rec_items=K_test,
        max_test_size=10_000,
        neg_sample_size=99
    )
    print(
        f"[NeuMF-pre TEST @ best epoch {neumf_best['epoch']}] "
        f"HR@{K_test}={hr_t:.4f}  NDCG@{K_test}={ndcg_t:.4f}"
    )


In [ ]:
# ----- Plots for pretrained NeuMF -----
EPOCHS_NEUMF = len(neumf_train_loss_hist)
epochs_neumf = list(range(1, EPOCHS_NEUMF + 1))
val_epochs_neumf = list(range(3, 3 * len(neumf_val_hr) + 1, 3))  # 3,6,...

# Training curves
plt.figure(figsize=(8, 5))
plt.plot(epochs_neumf, neumf_train_loss_hist, label='NeuMF-pre train_loss')
plt.plot(epochs_neumf, neumf_train_acc_hist, label='NeuMF-pre train_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('NeuMF (pretrained): Training Loss and Accuracy over Epochs')
plt.legend()
plt.tight_layout()
plt.show()

# Validation HR@K and NDCG@K every 3 epochs
plt.figure(figsize=(8, 5))
plt.plot(val_epochs_neumf, neumf_val_hr, marker='o',
         label=f'NeuMF-pre val_HR@{K_val}')
plt.plot(val_epochs_neumf, neumf_val_ndcg, marker='s',
         label=f'NeuMF-pre val_NDCG@{K_val}')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title(f'NeuMF (pretrained): Validation HR@{K_val} and NDCG@{K_val} over Epochs')
plt.legend()
plt.tight_layout()
plt.show()
